# Guangzhou Landsat Surface Temp Calculation

In [2]:
import ee
import geemap
import os
import rasterio
from rasterio.merge import merge

# Initialize Earth Engine
try:
    ee.Initialize(project='applied-spatial-rotterdam')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='applied-spatial-rotterdam')

# 1. Define the complete broad envelope coordinates
lon_min, lon_max = 112.8500, 114.1500
lat_min, lat_max = 22.5000, 23.9500

# Split the region into a 2x2 grid matrix (4 manageable tiles)
lon_mid = (lon_min + lon_max) / 2
lat_mid = (lat_min + lat_max) / 2

tile_coords = [
    ("south_west", [lon_min, lat_min, lon_mid, lat_mid]),
    ("south_east", [lon_mid, lat_min, lon_max, lat_mid]),
    ("north_west", [lon_min, lat_mid, lon_mid, lat_max]),
    ("north_east", [lon_mid, lat_mid, lon_max, lat_max])
]

# 2. Imagery Processing Engine
def get_processed_composite(tile_box):
    aoi = ee.Geometry.Rectangle(tile_box)
    spatial_filter = ee.Filter.bounds(aoi)
    date_filter = ee.Filter.date('2025-08-01', '2026-03-31')
    cloud_filter = ee.Filter.lt('CLOUD_COVER', 50)
    
    def mask_clouds(img):
        qa = img.select('QA_PIXEL')
        return img.updateMask(qa.bitwiseAnd(1 << 4).eq(0).And(qa.bitwiseAnd(1 << 3).eq(0)))
        
    def add_indicators(img):
        opt = img.select('SR_B.*').multiply(0.0000275).add(-0.2)
        ndvi = opt.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
        lst = img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST_Celsius')
        return img.addBands(opt, None, True).addBands(lst, None, True).addBands(ndvi)

    l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filter(spatial_filter).filter(date_filter).filter(cloud_filter)
    l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filter(spatial_filter).filter(date_filter).filter(cloud_filter)
    
    return l8.merge(l9).map(mask_clouds).map(add_indicators).median().clip(aoi)

# 3. Loop through tiles and download safely under the 50MB limit
os.makedirs('temp_tiles', exist_ok=True)
lst_tile_files = []
ndvi_tile_files = []

print("--- Downloading Sub-Tiles (Bypassing 50MB Cap) ---")
for name, box in tile_coords:
    print(f"Downloading tile: {name}...")
    tile_composite = get_processed_composite(box)
    aoi_geo = ee.Geometry.Rectangle(box)
    
    lst_file = f"temp_tiles/lst_{name}.tif"
    ndvi_file = f"temp_tiles/ndvi_{name}.tif"
    
    geemap.ee_export_image(tile_composite.select('LST_Celsius').toFloat(), filename=lst_file, scale=30, region=aoi_geo, file_per_band=False)
    geemap.ee_export_image(tile_composite.select('NDVI').toFloat(), filename=ndvi_file, scale=30, region=aoi_geo, file_per_band=False)
    
    lst_tile_files.append(lst_file)
    ndvi_tile_files.append(ndvi_file)

# 4. Local Mosaic Stitching Engine
print("\n--- Stitching Tiles Together Locally ---")
os.makedirs('../../data/processed', exist_ok=True)

def merge_and_save(file_list, output_path):
    src_files = [rasterio.open(f) for f in file_list]
    mosaic, out_trans = merge(src_files)
    out_meta = src_files[0].meta.copy()
    out_meta.update({
        "height": mosaic.shape[1], "width": mosaic.shape[2],
        "transform": out_trans, "crs": src_files[0].crs
    })
    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(mosaic)
    for src in src_files: src.close()

merge_and_save(lst_tile_files, '../../data/processed/guangzhou_lst_2025.tif')
merge_and_save(ndvi_tile_files, '../../data/processed/guangzhou_ndvi_2025.tif')
print("✔ Tiled stitching complete! Complete files saved to data/processed/")

--- Downloading Sub-Tiles (Bypassing 50MB Cap) ---
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\code\temp_tiles\lst_south_west.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\code\temp_tiles\ndvi_south_west.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\code\temp_tiles\lst_south_east.tif
Generating URL ...
Please wait ...


KeyboardInterrupt: 

In [ ]:
import geopandas as gpd
import osmnx as ox
import requests

# ============================================================
# FIX: Download proper Guangzhou subdistrict polygons from OSM
# (Replaces the original boundary-only guangzhou_admin.geojson
#  which contained only LineStrings with empty properties)
# ============================================================

print("Downloading Guangzhou admin boundaries from OSM...")

# Get the official Guangzhou city boundary as a clipping mask
guangzhou_boundary = ox.geocode_to_gdf("Guangzhou, Guangdong, China")
guangzhou_boundary = guangzhou_boundary.to_crs(epsg=4326)

# Download admin level 9 subdistricts (街道 / Jiedao level)
gdf_gz = ox.features_from_place(
    "Guangzhou, Guangdong, China",
    tags={"boundary": "administrative", "admin_level": "9"}
)

# Keep only polygon geometries
gdf_gz = gdf_gz[gdf_gz.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
gdf_gz = gdf_gz.reset_index(drop=True)
gdf_gz = gdf_gz.to_crs(epsg=4326)

# Clip strictly to Guangzhou city boundary to remove stray neighbouring districts
gdf_gz = gpd.clip(gdf_gz, guangzhou_boundary)

print(f"Found {len(gdf_gz)} subdistrict units within Guangzhou city boundary")

output = "../../data/Guangzhou/guangzhou_admin.geojson"
gdf_gz[['geometry', 'name']].to_file(output, driver="GeoJSON")
print(f"✔ Saved {len(gdf_gz)} polygon subdistricts to {output}")

In [ ]:
import geopandas as gpd
from rasterstats import zonal_stats

print("\n--- Running Zonal Statistics ---")
geojson_path = "../../data/Guangzhou/guangzhou_admin.geojson"
output_path  = "../../data/processed/guangzhou_admin_enriched.geojson"

gdf = gpd.read_file(geojson_path)
gdf['geometry'] = gdf['geometry'].make_valid()

# Extract values from our seamlessly stitched mosaics
gdf['mean_LST_celsius'] = [x['mean'] for x in zonal_stats(gdf, "../../data/processed/guangzhou_lst_2025.tif", stats=['mean'])]
gdf['mean_NDVI']        = [x['mean'] for x in zonal_stats(gdf, "../../data/processed/guangzhou_ndvi_2025.tif", stats=['mean'])]
gdf['majority_LCZ']     = [x['majority'] for x in zonal_stats(gdf, "../../data/processed/guangzhou_lcz_2018.tif", stats=['majority'])]

print(f"Missing LST values remaining: {gdf['mean_LST_celsius'].isna().sum()}")
print(f"Missing NDVI values remaining: {gdf['mean_NDVI'].isna().sum()}")

gdf.to_file(output_path, driver="GeoJSON")
print("✔ Enriched dataset saved successfully!")

In [ ]:
import geopandas as gpd

print("--- Final Cleanup: Dropping Edge Thermal Gaps & Water Artifacts ---")
enriched_path = "../../data/processed/guangzhou_admin_enriched.geojson"

# 1. Load the dataset we just exported
gdf = gpd.read_file(enriched_path)
initial_rows = len(gdf)

# 2. Drop rows missing thermal data
gdf_clean = gdf.dropna(subset=['mean_LST_celsius', 'mean_NDVI'])

# 3. Drop negative NDVI values (water bodies / cloud artifacts)
gdf_clean = gdf_clean[gdf_clean['mean_NDVI'] >= 0]

final_rows = len(gdf_clean)

print(f"-> Total districts before cleanup: {initial_rows}")
print(f"-> Total districts kept with full coverage: {final_rows}")
print(f"-> Successfully removed {initial_rows - final_rows} boundary gap / water rows.")

# 4. Overwrite the GeoJSON with the finalized, clean dataset
gdf_clean.to_file(enriched_path, driver="GeoJSON")
print("✔ Cleaned dataset safely saved! Ready for R clustering.")